# Exercise: Implementing Simple Input and Output Processors

Welcome! In this exercise, you'll get hands-on experience implementing the core components responsible for handling data as it enters and leaves a Large Language Model (LLM). This is inspired by the design of systems like vLLM, where `InputProcessors` and `OutputProcessors` manage the tokenization and detokenization lifecycle of a request.

By the end of this exercise, you will have:
*   Implemented a `SimpleInputProcessor` to convert a text prompt into a list of token IDs.
*   Implemented a `SimpleOutputProcessor` to convert lists of token IDs back into human-readable text.
*   Handled the logic for streaming outputs, where text is generated and decoded piece by piece.

Let's get started!

In [ ]:
import typing
from typing import List, Dict

# Helper Code - You don't need to modify this!
# We'll use a very simple character-level "tokenizer" for this exercise.
# This helps us focus on the processor logic without needing a complex library.

class SimpleTokenizer:
    """A basic character-level tokenizer for demonstration."""
    def __init__(self):
        # Create a simple vocabulary: a-z, space, and some punctuation.
        self.vocab = ["<unk>"] + list("abcdefghijklmnopqrstuvwxyz .,!?")
        self.char_to_id: Dict[str, int] = {char: i for i, char in enumerate(self.vocab)}
        self.id_to_char: Dict[int, str] = {i: char for i, char in enumerate(self.vocab)}
        self.unk_token_id = 0

    def encode(self, text: str) -> List[int]:
        """Converts a string of text into a list of token IDs."""
        text = text.lower()
        return [self.char_to_id.get(char, self.unk_token_id) for char in text]

    def decode(self, token_ids: List[int]) -> str:
        """Converts a list of token IDs back into a string."""
        return "".join([self.id_to_char.get(token_id, "") for token_id in token_ids])

# Let's instantiate our tokenizer. We'll use this for the rest of the exercise.
tokenizer = SimpleTokenizer()
print(f"Tokenizer vocabulary size: {len(tokenizer.vocab)}")
print(f"Encoding for 'hello': {tokenizer.encode('hello')}")
print(f"Decoding for [8, 5, 12, 12, 15]: {tokenizer.decode([8, 5, 12, 12, 15])}")

### Exercise 1: Implement the `SimpleInputProcessor`

Your first task is to implement the `SimpleInputProcessor`. This class has one job: to take a raw string prompt from a user and convert it into the list of token IDs that the LLM can understand.

**Instructions:**
*   Complete the `process_prompt` method.
*   The method should take a string `prompt` as input.
*   It should use the `self.tokenizer` object to encode the prompt.

**Hint:** The `SimpleTokenizer` class has an `encode()` method that does exactly what you need!

In [ ]:
class SimpleInputProcessor:
    """
    Processes a raw string prompt into a list of token IDs.
    """
    def __init__(self, tokenizer: SimpleTokenizer):
        self.tokenizer = tokenizer

    def process_prompt(self, prompt: str) -> List[int]:
        """
        Tokenizes a prompt string into a list of integer IDs.

        Args:
            prompt: The input string from the user.

        Returns:
            A list of token IDs representing the prompt.
        """

In [ ]:
# Test cell for Exercise 1
print("Running tests for SimpleInputProcessor...")

# 1. Set up the processor
input_processor = SimpleInputProcessor(tokenizer)
print("Input processor initialized.")

# 2. Test a simple prompt
prompt1 = "Hello world"
expected_ids1 = [8, 5, 12, 12, 15, 27, 23, 15, 18, 12, 4]
result_ids1 = input_processor.process_prompt(prompt1)
assert result_ids1 == expected_ids1, f"Test 1 Failed: Got {result_ids1}, expected {expected_ids1}"
print("✅ Test 1 passed: Simple prompt processed correctly.")

# 3. Test a prompt with punctuation
prompt2 = "How are you?"
expected_ids2 = [8, 15, 23, 27, 1, 18, 5, 27, 25, 15, 21, 30]
result_ids2 = input_processor.process_prompt(prompt2)
assert result_ids2 == expected_ids2, f"Test 2 Failed: Got {result_ids2}, expected {expected_ids2}"
print("✅ Test 2 passed: Prompt with punctuation processed correctly.")

# 4. Test an empty prompt
prompt3 = ""
expected_ids3 = []
result_ids3 = input_processor.process_prompt(prompt3)
assert result_ids3 == expected_ids3, f"Test 3 Failed: Got {result_ids3}, expected {expected_ids3}"
print("✅ Test 3 passed: Empty prompt processed correctly.")

print("\n🎉 All tests for SimpleInputProcessor passed!")

### Exercise 2: Implement the `SimpleOutputProcessor`

Great job! Now for the other side of the process. The `SimpleOutputProcessor` takes the token IDs generated by the model and converts them back into text.

A key challenge in LLM serving is **streaming**. The model generates tokens one by one (or in small chunks), and we want to show the output to the user as it's being generated, not just wait until the very end.

**Instructions:**
*   Complete the `process_stream_output` method.
*   This method receives the **entire sequence** of token IDs generated so far (`token_ids`).
*   It also receives the length of the sequence that has *already been processed* and sent to the user (`previous_ids_len`).
*   Your goal is to identify only the **new** tokens, decode them into a string, and return that new piece of text.

For example, if `token_ids` is `[8, 5, 12, 12, 15, 27, 9]` ("hello i") and `previous_ids_len` is `5`, it means the user has already seen "hello". Your function should identify the new tokens `[27, 9]` and return the decoded string `" i"`.

In [ ]:
class SimpleOutputProcessor:
    """
    Processes token IDs from the model into a human-readable string,
    handling streaming outputs.
    """
    def __init__(self, tokenizer: SimpleTokenizer):
        self.tokenizer = tokenizer

    def process_stream_output(self, token_ids: List[int], previous_ids_len: int) -> str:
        """
        Decodes only the new tokens generated since the last step.

        Args:
            token_ids: The full list of token IDs generated so far.
            previous_ids_len: The number of token IDs that have already been
                              processed and sent to the user.

        Returns:
            A string representing only the newly generated text.
        """

In [ ]:
# Test cell for Exercise 2
print("Running tests for SimpleOutputProcessor...")

# 1. Set up the processor
output_processor = SimpleOutputProcessor(tokenizer)
print("Output processor initialized.")

# 2. Simulate the first step of streaming
all_ids_step1 = [8, 5, 12, 12, 15] # "hello"
prev_len1 = 0
expected_text1 = "hello"
result_text1 = output_processor.process_stream_output(all_ids_step1, prev_len1)
assert result_text1 == expected_text1, f"Test 1 Failed: Got '{result_text1}', expected '{expected_text1}'"
print("✅ Test 1 passed: First chunk processed correctly.")

# 3. Simulate the second step of streaming
all_ids_step2 = [8, 5, 12, 12, 15, 27, 23, 15, 18, 12, 4] # "hello world"
prev_len2 = len(all_ids_step1) # length of the previous chunk
expected_text2 = " world"
result_text2 = output_processor.process_stream_output(all_ids_step2, prev_len2)
assert result_text2 == expected_text2, f"Test 2 Failed: Got '{result_text2}', expected '{expected_text2}'"
print("✅ Test 2 passed: Second chunk processed correctly.")

# 4. Simulate a step where no new tokens are generated
all_ids_step3 = [8, 5, 12, 12, 15, 27, 23, 15, 18, 12, 4] # Same as before
prev_len3 = len(all_ids_step3)
expected_text3 = ""
result_text3 = output_processor.process_stream_output(all_ids_step3, prev_len3)
assert result_text3 == expected_text3, f"Test 3 Failed: Got '{result_text3}', expected '{expected_text3}'"
print("✅ Test 3 passed: Empty chunk processed correctly.")

print("\n🎉 All tests for SimpleOutputProcessor passed!")

### Challenge: Handling Stop Sequences (Optional)

In a real system, we often want the model to stop generating text when it produces a specific word or phrase (e.g., "\n\n", "<|endoftext|>", "User:").

For an optional challenge, can you think about how you would modify `SimpleOutputProcessor` to handle this?
*   You might add a method like `set_stop_strings(self, stop_strings: List[str])`.
*   You would then need to modify `process_stream_output` to check if any of the new text (or the combination of old and new text) ends with a stop string.
*   If it does, it should return the text *up to but not including* the stop string, and perhaps return a signal indicating that generation should stop.

We won't be testing this, but it's a great way to think ahead to more advanced features!

In [ ]:
# Integration Test: Simulating a Full Streaming Request

Now let's put it all together! This final cell simulates a complete request lifecycle, from processing the input prompt to streaming the output token by token. You don't need to write any code here, just run the cell to see your `SimpleInputProcessor` and `SimpleOutputProcessor` working together.

print("Running integration test...")

# 1. Setup
my_prompt = "The quick brown fox"
input_processor = SimpleInputProcessor(tokenizer)
output_processor = SimpleOutputProcessor(tokenizer)

# 2. Input Processing
prompt_token_ids = input_processor.process_prompt(my_prompt)
print(f"Original Prompt: '{my_prompt}'")
print(f"Tokenized Prompt IDs: {prompt_token_ids}")
print("-" * 20)

# 3. Simulate Model Generation (we'll just add some token IDs manually)
generated_ids_to_add = tokenizer.encode(" jumps over the lazy dog.")
full_sequence_ids = prompt_token_ids + generated_ids_to_add

# 4. Output Processing (Streaming)
print("Streaming generated output:")
full_generated_text = ""
previous_len = len(prompt_token_ids)

# We'll "stream" two tokens at a time
for i in range(previous_len, len(full_sequence_ids), 2):
    current_len = min(i + 2, len(full_sequence_ids))
    # Get the chunk of text for the new tokens
    new_text_chunk = output_processor.process_stream_output(
        token_ids=full_sequence_ids[:current_len],
        previous_ids_len=previous_len
    )
    # Print and append
    print(new_text_chunk, end="", flush=True)
    full_generated_text += new_text_chunk
    # Update the length of what we've processed
    previous_len = current_len

print("\n" + "-" * 20)

# 5. Final Verification
expected_full_text = " jumps over the lazy dog."
assert full_generated_text == expected_full_text, f"Integration Test Failed: Got '{full_generated_text}', expected '{expected_full_text}'"

print("\n🎉 Congratulations! You have successfully implemented and integrated the input and output processors!")